# 12a — Annotation Sample Generation
### Runs on Colab (GPU required for BERT inference)
### **Run notebook 13 first** to generate the provenance mapping.

**What this notebook produces:**

| File | Purpose |
|------|---------|
| `annotation_sheets/model{N}_annotator_template.csv` | Send to each of the 4 annotators (one template per model) |
| `annotation_sheets/model{N}_master.csv` | Keep private — coordinator reference with Gemini & BERT labels |
| `annotation_sheets/model{N}_instructions.txt` | Send alongside the template CSV |

**Key change vs. previous version:** samples are drawn exclusively from sentences tagged `source == 'unarxive'` in the provenance mapping (notebook 13). Synthetic Gemini-generated sentences are excluded from human annotation.

**Notebook order:**
1. **Notebook 13** — build provenance mapping, publish to HF
2. **This notebook (12a)** — sample unarXive sentences, run BERT inference, export annotation sheets
3. Human annotation
4. **Notebook 12b** — compute kappa scores

## 1. Install Dependencies

In [1]:
!pip install -q datasets huggingface_hub
!pip install -q "tensorflow-text==2.15.*" "tf-models-official==2.15.*"

ERROR: Could not find a version that satisfies the requirement tensorflow-text==2.15.* (from versions: 2.18.1, 2.19.0rc0, 2.19.0, 2.20.0, 2.20.1)
ERROR: No matching distribution found for tensorflow-text==2.15.*


## 2. Imports

In [2]:
import os
import numpy as np
import pandas as pd
from sklearn import preprocessing
from sklearn.metrics import cohen_kappa_score

import tensorflow as tf
import tensorflow_text  # noqa
from datasets import load_dataset, Features, Value
from huggingface_hub import snapshot_download

tf.get_logger().setLevel('ERROR')
print('TF:', tf.__version__)

TF: 2.20.0


## 3. Constants

In [3]:
SEED              = 42
BATCH             = 32
SAMPLES_PER_CLASS = 10
ANNOTATION_DIR    = 'annotation_sheets'
DATASET_ID        = 'stormsidali2001/IMRAD-introduction-sentences-moves-sub-moves-dataset'

# Provenance mapping produced by notebook 13
PROVENANCE_ID     = 'stormsidali2001/IMRAD-introduction-sentences-unarxive-mapping'

MODEL_REPOS = {
    1: 'stormsidali2001/IMRAD_introduction_moves_classifier',
    2: 'stormsidali2001/IMRAD-introduction-move-zero-sub-moves-classifier',
    3: 'stormsidali2001/IMRAD-introduction-move-one-sub-moves-classifier',
    4: 'stormsidali2001/IMRAD-introduction-move-two-sub-moves-classifier',
}

MODEL_LABEL_SETS = {
    1: ['0', '1', '2'],
    2: ['0.0', '0.1'],
    3: ['1.0', '1.1', '1.2', '1.3'],
    4: ['2.0', '2.1', '2.2', '2.3', '2.4'],
}

MODEL_NAMES = {
    1: 'Model 1 - Overall Move Classifier (3 classes)',
    2: 'Model 2 - Move 0 Sub-move Classifier (2 classes)',
    3: 'Model 3 - Move 1 Sub-move Classifier (4 classes)',
    4: 'Model 4 - Move 2 Sub-move Classifier (5 classes)',
}

os.makedirs(ANNOTATION_DIR, exist_ok=True)
print(f'Output directory : {os.path.abspath(ANNOTATION_DIR)}')

Output directory : /content/annotation_sheets


## 4. Load Dataset

Force `move_sub_move_gemini` to string type to survive dirty rows (LaTeX tokens, malformed labels).

In [4]:
from datasets import load_dataset, Features, Value

features = Features({
    'sentence':             Value('string'),
    'move_sub_move_gemini': Value('string'),
})

# Load v3 main dataset
hf_ds  = load_dataset(DATASET_ID, split='train', features=features)
raw_df = hf_ds.to_pandas()
raw_df.insert(0, 'sentence_id', range(len(raw_df)))
print(f'v3 dataset loaded: {len(raw_df):,} sentences')

# Load provenance mapping from notebook 13
prov_hf = load_dataset(PROVENANCE_ID, split='train')
prov_df = prov_hf.to_pandas()[['sentence_id', 'source']]

# Merge and keep only unarXive sentences
raw_df = raw_df.merge(prov_df, on='sentence_id', how='left')
raw_df['source'] = raw_df['source'].fillna('synthetic')

full_df = raw_df[raw_df['source'] == 'unarxive'].copy()
print(f'After filtering to unarXive-only: {len(full_df):,} sentences')

def safe_int(x):
    try:
        return int(float(str(x).strip()))
    except (ValueError, TypeError):
        return -1

full_df['gemini_move'] = full_df['move_sub_move_gemini'].apply(safe_int)
full_df['sentence']    = full_df['sentence'].apply(lambda x: str(x).lower())
full_df = full_df[full_df['gemini_move'] != -1].copy()

print(f'Valid labelled sentences: {len(full_df):,}')
print(full_df['gemini_move'].value_counts().sort_index())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

dataset.csv:   0%|          | 0.00/75.1M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

v3 dataset loaded: 200,328 sentences


README.md:   0%|          | 0.00/486 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/16.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/200328 [00:00<?, ? examples/s]

After filtering to unarXive-only: 34,641 sentences
Valid labelled sentences: 8,796
gemini_move
0    3981
1    1488
2    3327
Name: count, dtype: int64


## 5. Reproduce Test Splits

Identical to notebooks 6–9 and 11: `random_state=42`, stratified by Gemini label. The LabelEncoder for sub-move models must be fit on **sorted** sub-move values to match the encoding used during training.

In [5]:
def make_split(df, seed=SEED):
    train     = df.sample(frac=0.8, random_state=seed)
    remainder = df.drop(train.index)
    val       = remainder.sample(frac=0.5, random_state=seed)
    test      = remainder.drop(val.index)
    return train, val, test


# Model 1 -- all 3 moves (Gemini label is already an int 0/1/2)
_, _, test1 = make_split(full_df)
le1 = None

# Model 2 -- Move 0 sub-moves (encoded 0->0.0, 1->0.1)
df2 = full_df[full_df['gemini_move'] == 0].sort_values('move_sub_move_gemini').copy()
le2 = preprocessing.LabelEncoder()
df2['encoded'] = le2.fit_transform(df2['move_sub_move_gemini'])
_, _, test2 = make_split(df2)

# Model 3 -- Move 1 sub-moves (encoded 0->1.0, 1->1.1, 2->1.2, 3->1.3)
df3 = full_df[full_df['gemini_move'] == 1].sort_values('move_sub_move_gemini').copy()
le3 = preprocessing.LabelEncoder()
df3['encoded'] = le3.fit_transform(df3['move_sub_move_gemini'])
_, _, test3 = make_split(df3)

# Model 4 -- Move 2 sub-moves (encoded 0->2.0 ... 4->2.4)
df4 = full_df[full_df['gemini_move'] == 2].sort_values('move_sub_move_gemini').copy()
le4 = preprocessing.LabelEncoder()
df4['encoded'] = le4.fit_transform(df4['move_sub_move_gemini'])
_, _, test4 = make_split(df4)

print(f'Test sizes  M1={len(test1)}  M2={len(test2)}  M3={len(test3)}  M4={len(test4)}')
for m, le in [(2, le2), (3, le3), (4, le4)]:
    print(f'  Model {m} encoder classes: {list(le.classes_)}')

Test sizes  M1=879  M2=398  M3=149  M4=333
  Model 2 encoder classes: ['0.0', '0.1']
  Model 3 encoder classes: ['1.0', '1.1', '1.2', '1.3']
  Model 4 encoder classes: ['2.0', '2.1', '2.2', '2.3', '2.4']


## 6. BERT Inference on Full Test Sets

Run all 4 models on their respective full test sets.

We need inference results for two things:
1. Attach `bert_pred_label` to the annotation sample (needed by notebook 12b to compute kappa without re-running BERT).
2. Compute the **full-test Gemini-Pro vs. BERT κ** right here, while predictions are already in memory.

In [6]:
def load_saved_model(repo_id):
    path   = snapshot_download(repo_id)
    loaded = tf.saved_model.load(path)
    infer  = loaded.signatures['serving_default']
    ik     = list(infer.structured_input_signature[1].keys())[0]
    ok     = list(infer.structured_outputs.keys())[0]
    print(f'  [{repo_id.split("/")[-1]}]  input={ik!r}  output={ok!r}')
    return infer, ik, ok


def batch_predict(infer, ik, ok, texts):
    out = []
    for i in range(0, len(texts), BATCH):
        b = tf.constant(texts[i:i + BATCH])
        out.append(infer(**{ik: b})[ok].numpy())
    return np.concatenate(out, axis=0)


def enc_to_str(enc_array, le):
    '''Map integer-encoded array to canonical string labels.'''
    if le is None:
        return np.array([str(int(x)) for x in enc_array])
    return np.array([str(le.classes_[int(x)]) for x in enc_array])

In [7]:
print('=== Loading and running all 4 models ===\n')

print('Model 1:')
infer1, ik1, ok1 = load_saved_model(MODEL_REPOS[1])
raw1        = batch_predict(infer1, ik1, ok1, test1.reset_index(drop=True)['sentence'].to_numpy())
y_pred1_enc = np.argmax(raw1, axis=1)
y_true1_enc = test1['gemini_move'].values

print('\nModel 2:')
infer2, ik2, ok2 = load_saved_model(MODEL_REPOS[2])
raw2        = batch_predict(infer2, ik2, ok2, test2.reset_index(drop=True)['sentence'].to_numpy())
y_pred2_enc = np.argmax(raw2, axis=1)
y_true2_enc = test2['encoded'].values

print('\nModel 3:')
infer3, ik3, ok3 = load_saved_model(MODEL_REPOS[3])
raw3        = batch_predict(infer3, ik3, ok3, test3.reset_index(drop=True)['sentence'].to_numpy())
y_pred3_enc = np.argmax(raw3, axis=1)
y_true3_enc = test3['encoded'].values

print('\nModel 4:')
infer4, ik4, ok4 = load_saved_model(MODEL_REPOS[4])
raw4        = batch_predict(infer4, ik4, ok4, test4.reset_index(drop=True)['sentence'].to_numpy())
y_pred4_enc = np.argmax(raw4, axis=1)
y_true4_enc = test4['encoded'].values

print('\nInference complete.')

=== Loading and running all 4 models ===

Model 1:


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

  [IMRAD_introduction_moves_classifier]  input='text'  output='classifier'

Model 2:


Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

  [IMRAD-introduction-move-zero-sub-moves-classifier]  input='text'  output='classifier'

Model 3:


Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

  [IMRAD-introduction-move-one-sub-moves-classifier]  input='text'  output='classifier'

Model 4:


Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

  [IMRAD-introduction-move-two-sub-moves-classifier]  input='text'  output='classifier'

Inference complete.


### 6.1 Full-Test Gemini-Pro vs. BERT Kappa

**Why compute this here?**  
The dataset carries a Gemini Pro label for every sentence (`move_sub_move_gemini`). We just ran BERT inference on the full test sets, so we now have both labels for ~5k–17k sentences per model — at no extra cost.

For the three system pairs (Human vs. Gemini, Human vs. BERT, Gemini vs. BERT), only the Gemini-vs-BERT pair can be computed without human annotation. Computing it on the full test set (~16k sentences for Model 1) gives a statistically robust number that complements the smaller annotated-subset kappas computed in notebook 12b.

In [8]:
print('Full-test Gemini-Pro vs. BERT  |  Cohen\'s kappa')
print('=' * 60)

full_test_configs = {
    1: (y_true1_enc, y_pred1_enc, None, len(test1)),
    2: (y_true2_enc, y_pred2_enc, le2,  len(test2)),
    3: (y_true3_enc, y_pred3_enc, le3,  len(test3)),
    4: (y_true4_enc, y_pred4_enc, le4,  len(test4)),
}

full_test_kappas = {}
for mid, (y_true, y_pred, le, n) in full_test_configs.items():
    ls  = MODEL_LABEL_SETS[mid]
    a   = enc_to_str(y_true, le)
    b   = enc_to_str(y_pred, le)
    k   = cohen_kappa_score(a, b, labels=ls)
    full_test_kappas[mid] = k
    print(f'  {MODEL_NAMES[mid]:<60s}  n={n:6d}  k={k:+.4f}')

print('\nThese values will also be saved into the master CSV for reference in notebook 12b.')

Full-test Gemini-Pro vs. BERT  |  Cohen's kappa
  Model 1 - Overall Move Classifier (3 classes)                 n=   879  k=+0.9076
  Model 2 - Move 0 Sub-move Classifier (2 classes)              n=   398  k=+0.8342
  Model 3 - Move 1 Sub-move Classifier (4 classes)              n=   149  k=+0.7988
  Model 4 - Move 2 Sub-move Classifier (5 classes)              n=   333  k=+0.8565

These values will also be saved into the master CSV for reference in notebook 12b.


## 7. Sample 10 Sentences per Class

Stratified sample from the test set, one per Gemini label class. The sample is **shuffled** after stratification so annotators cannot detect class order from row position.

| Model | Classes | Sentences sampled |
|-------|---------|-------------------|
| 1 | 3 | 30 |
| 2 | 2 | 20 |
| 3 | 4 | 40 |
| 4 | 5 | 50 |

In [9]:
def sample_test_set(test_df, y_true_enc, y_pred_enc, le=None,
                    n=SAMPLES_PER_CLASS, seed=SEED):
    """
    Stratified sample of n sentences per Gemini-label class from the test set.
    Returns a DataFrame with id, sentence, gemini_label, bert_pred_label.
    """
    df = test_df.reset_index(drop=True).copy()
    df['_y_true'] = y_true_enc
    df['_y_pred'] = y_pred_enc
    df['gemini_label']    = enc_to_str(df['_y_true'].values, le)
    df['bert_pred_label'] = enc_to_str(df['_y_pred'].values, le)

    sampled = (
        df.groupby('_y_true', group_keys=False)
          .apply(lambda g: g.sample(min(n, len(g)), random_state=seed))
          .sample(frac=1, random_state=seed)  # shuffle to hide class order
          .reset_index(drop=True)
    )
    sampled.insert(0, 'id', range(1, len(sampled) + 1))
    sampled['sentence'] = sampled['sentence'].str.strip()
    return sampled[['id', 'sentence', 'gemini_label', 'bert_pred_label']]

In [10]:
sample_m1 = sample_test_set(test1, y_true1_enc, y_pred1_enc, le=None)
sample_m2 = sample_test_set(test2, y_true2_enc, y_pred2_enc, le=le2)
sample_m3 = sample_test_set(test3, y_true3_enc, y_pred3_enc, le=le3)
sample_m4 = sample_test_set(test4, y_true4_enc, y_pred4_enc, le=le4)

for mid, sdf in [(1, sample_m1), (2, sample_m2), (3, sample_m3), (4, sample_m4)]:
    dist = sdf.groupby('gemini_label').size().to_dict()
    print(f'Model {mid}: {len(sdf)} sentences  {dist}')

Model 1: 30 sentences  {'0': 10, '1': 10, '2': 10}
Model 2: 20 sentences  {'0.0': 10, '0.1': 10}
Model 3: 39 sentences  {'1.0': 10, '1.1': 10, '1.2': 9, '1.3': 10}
Model 4: 50 sentences  {'2.0': 10, '2.1': 10, '2.2': 10, '2.3': 10, '2.4': 10}


/tmp/ipykernel_2092/3932779554.py:15: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(min(n, len(g)), random_state=seed))
/tmp/ipykernel_2092/3932779554.py:15: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(min(n, len(g)), random_state=seed))
/tmp/ipykernel_2092/3932779554.py:15: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. Thi

## 8. Save Annotation CSVs

Two types of output per model:

**Annotator template** (`model{N}_annotator_template.csv`)  
What you send to annotators. Contains `id`, `sentence`, and an empty `your_label` column.  
All 4 annotators receive the same template file and fill it in independently.

**Master reference** (`model{N}_master.csv`)  
Coordinator-only. Contains `gemini_label`, `bert_pred_label`, and empty `annotator_1..4` columns to be filled by copying from returned annotator sheets.

In [11]:
def save_annotation_files(sample_df, model_id, full_test_kappa,
                          out_dir=ANNOTATION_DIR):
    label_set = MODEL_LABEL_SETS[model_id]

    # 1. Annotator template (no reference labels -- blind)
    template = sample_df[['id', 'sentence']].copy()
    template['valid_labels'] = str(label_set)
    template['your_label']   = ''
    template.to_csv(f'{out_dir}/model{model_id}_annotator_template.csv', index=False)

    # 2. Master reference (gemini + bert labels, columns for 4 annotators)
    master = sample_df.copy()
    master['full_test_gemini_vs_bert_kappa'] = round(full_test_kappa, 4)
    for ann in range(1, 5):
        master[f'annotator_{ann}'] = ''
    master.to_csv(f'{out_dir}/model{model_id}_master.csv', index=False)

    print(f'Model {model_id}:')
    print(f'  -> {out_dir}/model{model_id}_annotator_template.csv  ({len(template)} rows)')
    print(f'  -> {out_dir}/model{model_id}_master.csv')


print('Saving annotation files...\n')
save_annotation_files(sample_m1, 1, full_test_kappas[1])
save_annotation_files(sample_m2, 2, full_test_kappas[2])
save_annotation_files(sample_m3, 3, full_test_kappas[3])
save_annotation_files(sample_m4, 4, full_test_kappas[4])

Saving annotation files...

Model 1:
  -> annotation_sheets/model1_annotator_template.csv  (30 rows)
  -> annotation_sheets/model1_master.csv
Model 2:
  -> annotation_sheets/model2_annotator_template.csv  (20 rows)
  -> annotation_sheets/model2_master.csv
Model 3:
  -> annotation_sheets/model3_annotator_template.csv  (39 rows)
  -> annotation_sheets/model3_master.csv
Model 4:
  -> annotation_sheets/model4_annotator_template.csv  (50 rows)
  -> annotation_sheets/model4_master.csv


## 9. Save Annotation Instruction Files

One `.txt` file per model. Distribute alongside the annotator template CSV.

In [12]:
def save_instructions(model_id, lines, out_dir=ANNOTATION_DIR):
    path = f'{out_dir}/model{model_id}_instructions.txt'
    with open(path, 'w') as f:
        f.write('\n'.join(lines) + '\n')
    print(f'Saved: {path}')


save_instructions(1, [
    'ANNOTATION TASK -- Model 1 (Overall Move Classifier)',
    '=' * 52,
    '',
    'Read each sentence in the CSV. In the "your_label" column, write',
    'the number (0, 1, or 2) that best matches its rhetorical function.',
    '',
    'VALID LABELS:',
    '',
    '  0   Move 0: Establishing a Research Territory',
    '      The sentence shows the topic is important/relevant, or reviews prior research.',
    '      Examples:',
    '        "Evidence suggests that X is among the most important factors for ..."',
    '        "Extensive research has shown that ..."',
    '',
    '  1   Move 1: Establishing a Niche',
    '      The sentence identifies a gap, flaw, or limitation in existing research.',
    '      Examples:',
    '        "However, little research has addressed ..."',
    '        "These methods fail to consider ..."',
    '',
    '  2   Move 2: Occupying the Niche',
    '      The sentence states what the paper does: aims, findings, structure.',
    '      Examples:',
    '        "This paper aims to ..."',
    '        "Our results show that ..."',
])

save_instructions(2, [
    'ANNOTATION TASK -- Model 2 (Move 0 Sub-move Classifier)',
    '=' * 55,
    '',
    'All sentences belong to Move 0 (Establishing a Research Territory).',
    'In the "your_label" column, write either 0.0 or 0.1 (write exactly as shown).',
    '',
    'VALID LABELS:',
    '',
    '  0.0   Show that the topic is important / relevant',
    '        Claims importance, prevalence, or relevance WITHOUT primarily',
    '        summarising specific prior studies.',
    '        Examples:',
    '          "Evidence suggests that X is among the most important factors for ..."',
    '          "Existing research recognises the critical role played by ..."',
    '',
    '  0.1   Introduce and review previous research in the field',
    '        Mentions, cites, or summarises specific prior studies.',
    '        Examples:',
    '          "Extensive research has shown that ..."',
    '          "Several studies have explored ..."',
])

save_instructions(3, [
    'ANNOTATION TASK -- Model 3 (Move 1 Sub-move Classifier)',
    '=' * 55,
    '',
    'All sentences belong to Move 1 (Establishing a Niche).',
    'In the "your_label" column, write one of: 1.0  1.1  1.2  1.3 (exactly as shown).',
    '',
    'VALID LABELS:',
    '',
    '  1.0   Claim a flaw in prior work',
    '        Directly criticises a problem in prior research.',
    '        Example: "However, these methods fail to consider ..."',
    '',
    '  1.1   Highlight a gap in the field',
    '        Notes that something has not been studied or is insufficient.',
    '        Example: "Little research has addressed ..."',
    '',
    '  1.2   Raise an unclear or unresolved question',
    '        Notes that something is unclear or poorly understood.',
    '        Example: "The mechanisms underlying X remain unclear ..."',
    '',
    '  1.3   Extend prior research',
    '        Proposes to build on or extend existing research.',
    '        Example: "This paper extends the work of ... by ..."',
])

save_instructions(4, [
    'ANNOTATION TASK -- Model 4 (Move 2 Sub-move Classifier)',
    '=' * 55,
    '',
    'All sentences belong to Move 2 (Occupying the Niche).',
    'In the "your_label" column, write one of: 2.0  2.1  2.2  2.3  2.4 (exactly as shown).',
    '',
    'VALID LABELS:',
    '',
    '  2.0   State the purpose / aim of the research',
    '        Describes what the paper intends to do.',
    '        Example: "This paper aims to ... / The goal of this work is ..."',
    '',
    '  2.1   State the hypothesis or research question',
    '        Presents an expected result or a formal question.',
    '        Example: "We hypothesise that ... / We investigate whether ..."',
    '',
    '  2.2   Share findings / contributions',
    '        Describes what was found, built, or achieved.',
    '        Example: "Our results show ... / We propose a method that ..."',
    '',
    '  2.3   Elaborate on the value of the research',
    '        Explains why the findings matter.',
    '        Example: "These results contribute to ... / This is significant because ..."',
    '',
    '  2.4   Outline the paper structure',
    '        Describes how the rest of the paper is organised.',
    '        Example: "The rest of this paper is structured as follows ..."',
])

Saved: annotation_sheets/model1_instructions.txt
Saved: annotation_sheets/model2_instructions.txt
Saved: annotation_sheets/model3_instructions.txt
Saved: annotation_sheets/model4_instructions.txt


## 10. Download from Colab

Run this cell to zip and download the full `annotation_sheets/` folder.

In [13]:
import shutil
from google.colab import files

zip_path = 'annotation_sheets'
shutil.make_archive(zip_path, 'zip', zip_path)
files.download(f'{zip_path}.zip')

print('Downloaded annotation_sheets.zip')
print()
print('Contents:')
for f in sorted(os.listdir(ANNOTATION_DIR)):
    print(f'  {f}')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded annotation_sheets.zip

Contents:
  model1_annotator_template.csv
  model1_instructions.txt
  model1_master.csv
  model2_annotator_template.csv
  model2_instructions.txt
  model2_master.csv
  model3_annotator_template.csv
  model3_instructions.txt
  model3_master.csv
  model4_annotator_template.csv
  model4_instructions.txt
  model4_master.csv


---

## Next Steps

1. Unzip `annotation_sheets.zip`.
2. For **each model N (1, 2, 3, 4)**, send these two files to **each** of your 4 annotators:
   - `model{N}_annotator_template.csv` — they fill the `your_label` column
   - `model{N}_instructions.txt` — explains what each label means
3. Collect the 4 returned CSVs per model and rename them:
   ```
   model1_annotator1_completed.csv
   model1_annotator2_completed.csv
   model1_annotator3_completed.csv
   model1_annotator4_completed.csv
   model2_annotator1_completed.csv
   ... (16 files total)
   ```
4. Place all completed CSVs and the `model{N}_master.csv` files in the same folder.
5. Open **notebook 12b** and run it to compute all kappa scores.